# 04. DEG可視化DEG結果をインタラクティブに可視化します。**カーネル**: RNA-seq (Python) / rnaseq_env  **入力**: `results/deg_*.csv`  **出力**: Volcano Plot, 全条件比較ヒートマップ (HTML)### 色の定義- **赤色**: 有意に増加 (log2FC > 1.0 & padj < 0.05)- **グレー**: 変化なし (padj >= 0.05)- **青色**: 有意に減少 (log2FC < -1.0 & padj < 0.05)

## 設定

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
import glob, os

# === 設定 ===
LFC_THRESHOLD = 1.0
PADJ_THRESHOLD = 0.05

# 色の定義
COLOR_UP = "#E74C3C"     # 赤: 増加
COLOR_DOWN = "#3498DB"   # 青: 減少
COLOR_NS = "#BDC3C7"     # グレー: 変化なし

print("設定完了")

## データ読み込みとDEG分類

In [ ]:
# 全比較結果を読み込み
result_files = sorted(glob.glob("results/deg_*_vs_*.csv"))
all_results = {}

for f in result_files:
    df = pd.read_csv(f)
    comp_name = df["comparison"].iloc[0]
    all_results[comp_name] = df
    
print(f"比較条件数: {len(all_results)}")
for name, df in all_results.items():
    print(f"  {name}: {len(df)} genes")

In [ ]:
def classify_deg(row):
    if pd.isna(row["padj"]) or pd.isna(row["log2FoldChange"]):
        return "NS"
    if row["padj"] < PADJ_THRESHOLD and row["log2FoldChange"] > LFC_THRESHOLD:
        return "Up"
    elif row["padj"] < PADJ_THRESHOLD and row["log2FoldChange"] < -LFC_THRESHOLD:
        return "Down"
    return "NS"

for comp_name, df in all_results.items():
    df["deg_class"] = df.apply(classify_deg, axis=1)
    n_up = (df["deg_class"] == "Up").sum()
    n_down = (df["deg_class"] == "Down").sum()
    n_ns = (df["deg_class"] == "NS").sum()
    print(f"{comp_name}: Up={n_up}, Down={n_down}, NS={n_ns}")

## Volcano Plot（インタラクティブ・ドロップダウン切替）

In [ ]:
fig = go.Figure()
comp_names = list(all_results.keys())

for i, (comp_name, df) in enumerate(all_results.items()):
    df = df.dropna(subset=["log2FoldChange", "padj"]).copy()
    df["-log10padj"] = -np.log10(df["padj"].clip(lower=1e-300))
    
    for deg_class, color in [("NS", COLOR_NS), ("Down", COLOR_DOWN), ("Up", COLOR_UP)]:
        subset = df[df["deg_class"] == deg_class]
        fig.add_trace(go.Scattergl(
            x=subset["log2FoldChange"],
            y=subset["-log10padj"],
            mode="markers",
            marker=dict(color=color, size=4, opacity=0.6),
            name=deg_class,
            text=subset["gene_id"],
            hovertemplate="<b>%{text}</b><br>log2FC: %{x:.2f}<br>-log10(padj): %{y:.2f}<extra></extra>",
            visible=(i == 0),
            legendgroup=deg_class,
            showlegend=(i == 0)
        ))

# ドロップダウン
buttons = []
for i, name in enumerate(comp_names):
    vis = [False] * (len(comp_names) * 3)
    for j in range(3):
        vis[i * 3 + j] = True
    buttons.append(dict(label=name, method="update",
                        args=[{"visible": vis}, {"title": f"Volcano Plot: {name}"}]))

# 閾値線
fig.add_hline(y=-np.log10(PADJ_THRESHOLD), line_dash="dash", line_color="grey", opacity=0.5)
fig.add_vline(x=LFC_THRESHOLD, line_dash="dash", line_color="grey", opacity=0.5)
fig.add_vline(x=-LFC_THRESHOLD, line_dash="dash", line_color="grey", opacity=0.5)

fig.update_layout(
    updatemenus=[dict(type="dropdown", x=0.0, y=1.15, showactive=True, buttons=buttons)],
    title=f"Volcano Plot: {comp_names[0]}",
    xaxis_title="log2 Fold Change",
    yaxis_title="-log10(adjusted p-value)",
    template="plotly_white", width=900, height=700,
    legend=dict(title="DEG Classification")
)

fig.show()

In [ ]:
# HTML出力
fig.write_html("results/volcano_plot_interactive.html", include_plotlyjs=True)
print("出力: results/volcano_plot_interactive.html")

## 全条件比較ヒートマップ全ての比較条件で、各遺伝子のDEG状態を一覧表示します。- **赤色**: 増加 (Up-regulated)- **グレー**: 変化なし (Not Significant)- **青色**: 減少 (Down-regulated)

In [ ]:
# DEG分類行列の構築
deg_matrices = {}
lfc_matrices = {}

for comp_name, df in all_results.items():
    df_idx = df.set_index("gene_id")
    deg_matrices[comp_name] = df_idx["deg_class"].map({"Up": 1, "NS": 0, "Down": -1}).fillna(0)
    lfc_matrices[comp_name] = df_idx["log2FoldChange"].fillna(0)

deg_df = pd.DataFrame(deg_matrices)
lfc_df = pd.DataFrame(lfc_matrices)

# いずれかの比較でDEGだった遺伝子のみ
is_deg = (deg_df.abs() > 0).any(axis=1)
deg_df = deg_df[is_deg]
lfc_df = lfc_df.loc[deg_df.index]

print(f"DEG遺伝子数 (いずれかの条件): {len(deg_df)}")

In [ ]:
# クラスタリング
if len(deg_df) > 1:
    dist_matrix = pdist(deg_df.values, metric="euclidean")
    link = linkage(dist_matrix, method="ward")
    order = leaves_list(link)
    deg_df = deg_df.iloc[order]
    lfc_df = lfc_df.iloc[order]

# ホバーテキスト
hover = []
for gene in deg_df.index:
    row = []
    for comp in deg_df.columns:
        s = {1: "Up", 0: "NS", -1: "Down"}.get(deg_df.loc[gene, comp], "NS")
        row.append(f"Gene: {gene}<br>Comparison: {comp}<br>Status: {s}<br>log2FC: {lfc_df.loc[gene, comp]:.2f}")
    hover.append(row)

# カラースケール: 青(-1) → グレー(0) → 赤(+1)
colorscale = [
    [0.0, COLOR_DOWN], [0.45, COLOR_DOWN],
    [0.45, COLOR_NS], [0.55, COLOR_NS],
    [0.55, COLOR_UP], [1.0, COLOR_UP]
]

fig_h = max(600, min(2000, len(deg_df) * 12 + 200))

fig_heat = go.Figure(data=go.Heatmap(
    z=deg_df.values,
    x=deg_df.columns.tolist(),
    y=deg_df.index.tolist(),
    colorscale=colorscale,
    zmin=-1, zmax=1,
    text=hover, hoverinfo="text",
    colorbar=dict(title="DEG Status", tickvals=[-1, 0, 1], ticktext=["Down", "NS", "Up"], len=0.5)
))

fig_heat.update_layout(
    title="全条件比較DEGヒートマップ",
    xaxis_title="比較条件",
    yaxis_title="遺伝子",
    template="plotly_white",
    width=max(600, len(deg_df.columns) * 150 + 300),
    height=fig_h,
    yaxis=dict(tickfont=dict(size=8 if len(deg_df) > 50 else 10))
)

fig_heat.show()

In [ ]:
# HTML出力
fig_heat.write_html("results/deg_heatmap_interactive.html", include_plotlyjs=True)
print("出力: results/deg_heatmap_interactive.html")

## DEG数サマリー (バープロット)

In [ ]:
counts_list = []
for comp_name, df in all_results.items():
    n_up = (df["deg_class"] == "Up").sum()
    n_down = (df["deg_class"] == "Down").sum()
    counts_list.append({"comparison": comp_name, "Up": n_up, "Down": -n_down})

count_df = pd.DataFrame(counts_list)

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(x=count_df["comparison"], y=count_df["Up"],
                         name="Up-regulated", marker_color=COLOR_UP))
fig_bar.add_trace(go.Bar(x=count_df["comparison"], y=count_df["Down"],
                         name="Down-regulated", marker_color=COLOR_DOWN))

fig_bar.update_layout(
    title="DEG数 (各比較条件)",
    xaxis_title="比較", yaxis_title="遺伝子数",
    barmode="relative", template="plotly_white"
)
fig_bar.show()

## 完了### 生成されたファイル- `results/volcano_plot_interactive.html` — インタラクティブVolcano Plot- `results/deg_heatmap_interactive.html` — 全条件比較ヒートマップHTMLファイルをブラウザで開くと、インタラクティブに操作できます。### 操作方法- **Volcano Plot**: ドロップダウンで比較条件を切替。マウスホバーで遺伝子名表示。- **ヒートマップ**: マウスホバーで遺伝子名・条件・log2FC・DEGステータスを表示。ズーム・パン可能。